In [1]:
import numpy as np
import pandas as pd
import os
import joblib
from datetime import datetime
import gradio as gr

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# -----------------------
# CONFIGURACIÓN DE RUTAS
# -----------------------
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/hackathon/'
#DATASET_PRINCIPAL = BASE_PATH + 'dataset_modelado_sentimiento.csv'
DATASET_FEEDBACK = BASE_PATH + 'consultas_recolectadas.csv'
MODEL_PATH = BASE_PATH + 'modelo_final.pkl'
VECT_PATH = BASE_PATH + 'vectorizador_final.pkl'

# --------------------------------------------
# CARGA Y UNIFICACIÓN DE DATOS
# --------------------------------------------
def cargar_datos_unificados():
    df_base = pd.read_csv(DATASET_PRINCIPAL)
    if 'Frase' in df_base.columns:
        df_base.rename(columns={'Frase': 'clean_text', 'Etiqueta': 'y'}, inplace=True)

    if os.path.exists(DATASET_FEEDBACK):
        df_feedback = pd.read_csv(DATASET_FEEDBACK)
        df_feedback_clean = df_feedback[['clean_text', 'y']]
        df_unificado = pd.concat([df_base, df_feedback_clean], ignore_index=True)
        print(f"✅ Entrenando con {len(df_base)} base + {len(df_feedback)} correcciones nuevas.")
    else:
        df_unificado = df_base

    df_unificado.dropna(subset=['clean_text', 'y'], inplace=True)
    df_unificado['y'] = df_unificado['y'].astype(int)
    return df_unificado

#df = cargar_datos_unificados()



# -----------------------------------
# ENTRENAMIENTO OPTIMIZADO
# -----------------------------------
"""
X = df['clean_text'].fillna('')
y = df['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

stop_words_es = [
    "el", "la", "los", "las", "un", "una", "unos", "unas", "de", "en", "con", "para", "por",
    "sobre", "entre", "y", "o", "que", "porque", "como", "cuando", "donde", "a", "al", "del",
    "es", "son", "fue", "eran", "ser", "estar", "tener", "hay", "esta", "está", "sido", "ir", "va",
    "yo", "tu", "él", "ella", "nosotros", "ustedes", "ellos", "mi", "mis", "su", "sus", "me", "se", "le",
    "sevilla", "hotel", "hostal", "alojamiento", "estancia", "noche", "noches",
    "habitación", "habitacion", "camas", "cama", "baño", "desayuno", "ubicación",
    "situación", "precio", "check", "in", "out", "personal", "cliente", "recepción"
]


vectorizer = TfidfVectorizer(stop_words=stop_words_es, ngram_range=(1, 2), max_df=0.9, min_df=5, lowercase=True, sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(X_train)
model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train_vec, y_train)

joblib.dump(model, MODEL_PATH)
joblib.dump(vectorizer, VECT_PATH)
"""

print("🤖 Activando ROBOT DE INTELIGENCIA REFORZADO...")

ROBOT_PATH = "/content/R5K_v2.pkl"  # Ruta a tu nuevo modelo reforzado

pipeline_robot = joblib.load(ROBOT_PATH)

vectorizer = pipeline_robot.named_steps['tfidf']
model = pipeline_robot.named_steps['clf']

print("✅ Robot reforzado R5K_v2 conectado y operativo")

# ------------------------------------------------
# LÓGICA DE LA MÁSCARA (CON AUTO-APRENDIZAJE CONTROLADO)
# ------------------------------------------------
def aplicar_regla_linguistica(texto, prob_pos):
    texto_limpio = texto.lower()

    patrones_negativos = [
        "desordenado", "muy desordenado", "caótico", "sucio",
        "mal organizado", "desorganizado", "desastre", "horrible",
        "pésimo", "fatal", "terrible"
    ]

    penalizacion_total = 0.0

    for p in patrones_negativos:
        if p in texto_limpio:
            penalizacion_total += 0.45  # penalización fuerte pero no absoluta

    prob_corregida = prob_pos - penalizacion_total

    # límites de seguridad
    prob_corregida = max(0.05, min(prob_corregida, 0.95))

    return prob_corregida

def predecir_solo_muestra(texto):
    """Analiza el texto pero NO guarda nada en el CSV todavía."""
    if not texto.strip():
        return {}, "⚠️ Por favor, escribe una opinión primero."

    vec = vectorizer.transform([texto])
    prob_pos = float(model.predict_proba(vec)[0][1])
    prob_pos = aplicar_regla_linguistica(texto, prob_pos)

    dict_grafico = {"POSITIVO 😊 ✅": prob_pos, "NEGATIVO 😟 ❌": 1 - prob_pos}

    return dict_grafico, "### 🔍 Análisis listo. ¿Es correcto? Selecciona una opción abajo para guardar."

def confirmar_y_guardar(texto, dict_grafico):
    """Guarda la etiqueta que la IA predijo originalmente."""
    if not texto.strip() or not dict_grafico:
        return "⚠️ No hay análisis previo para confirmar."

    prob_pos = dict_grafico.get("POSITIVO 😊 ✅", 0)
    etiqueta = 1 if prob_pos > 0.5 else 0

    nuevo_dato = pd.DataFrame([[texto, etiqueta, datetime.now()]], columns=['clean_text', 'y', 'fecha_recoleccion'])
    header = not os.path.exists(DATASET_FEEDBACK)
    nuevo_dato.to_csv(DATASET_FEEDBACK, mode='a', header=header, index=False)

    return f"### ✅ Confirmado como {'POSITIVO' if etiqueta == 1 else 'NEGATIVO'} y guardado en el historial."

def corregir_y_guardar(texto, dict_grafico):
    """Guarda la etiqueta OPUESTA a la que predijo la IA."""
    if not texto.strip() or not dict_grafico:
        return "⚠️ No hay análisis previo para corregir."

    prob_pos_original = dict_grafico.get("POSITIVO 😊 ✅", 0)
    etiqueta_corregida = 0 if prob_pos_original > 0.5 else 1

    nuevo_dato = pd.DataFrame([[texto, etiqueta_corregida, datetime.now()]], columns=['clean_text', 'y', 'fecha_recoleccion'])
    header = not os.path.exists(DATASET_FEEDBACK)
    nuevo_dato.to_csv(DATASET_FEEDBACK, mode='a', header=header, index=False)

    return f"### 🛠️ ¡Corregido! Guardado manualmente como {'POSITIVO' if etiqueta_corregida == 1 else 'NEGATIVO'}."

def procesar_archivo_masivo_automatizado(archivo):
    try:
        df_m = pd.read_csv(archivo.name) if archivo.name.endswith('.csv') else pd.read_excel(archivo.name)
        col_texto = 'clean_text' if 'clean_text' in df_m.columns else df_m.columns[0]

        textos = df_m[col_texto].fillna("").astype(str)
        vecs = vectorizer.transform(textos)

        # Obtenemos probabilidades para decidir qué automatizar
        probs = model.predict_proba(vecs)
        preds = model.predict(vecs)

        # Umbral de confianza (85%)
        UMBRAL = 0.85

        registros_para_guardar = []
        automaticos = 0
        para_revision = 0

        for i, (p, pred) in enumerate(zip(probs, preds)):
            confianza = max(p)
            texto = textos.iloc[i]

            if confianza >= UMBRAL:
                # Si la IA está muy segura, lo guarda automáticamente en el feedback
                registros_para_guardar.append([texto, pred, datetime.now()])
                automaticos += 1
            else:
                para_revision += 1

        # Guardado masivo automático de registros seguros
        if registros_para_guardar:
            df_auto = pd.DataFrame(registros_para_guardar, columns=['clean_text', 'y', 'fecha_recoleccion'])
            header = not os.path.exists(DATASET_FEEDBACK)
            df_auto.to_csv(DATASET_FEEDBACK, mode='a', header=header, index=False)

        # Preparar archivo de salida para el usuario
        df_m['Sentimiento_IA'] = ["POSITIVO" if p == 1 else "NEGATIVO" for p in preds]
        df_m['Confianza'] = [f"{round(max(p)*100)}%" for p in probs]

        output_file = "/content/resultado_analisis_SAPI.xlsx"
        df_m.to_excel(output_file, index=False)

        resumen = f"🚀 AUTOMATIZACIÓN COMPLETADA:\n\n" \
                  f"• ✅ {automaticos} registros guardados automáticamente (Confianza > {UMBRAL*100}%).\n" \
                  f"• ⚠️ {para_revision} registros procesados pero NO guardados (requieren revisión humana).\n" \
                  f"• 📊 Total procesado: {len(df_m)}."

        return output_file, resumen
    except Exception as e:
        return None, f"❌ Error: {str(e)}"

# ------------------------------------------------
# INTERFAZ GRADIO OPTIMIZADA (V2 - CONFIRMACIÓN EXPLÍCITA)
# ------------------------------------------------
def limpiar_pantalla():
    return "", {}, ""
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Central de Inteligencia de Sentimientos")

    with gr.Tabs():
        # PESTAÑA 1: ANÁLISIS INDIVIDUAL
        with gr.TabItem("🎯 Análisis Individual"):
            with gr.Row():
                with gr.Column():
                    input_text = gr.Textbox(label="Escribe la opinión aquí", lines=6, placeholder="Ej: El servicio fue pésimo...")
                    btn_analizar = gr.Button("🔍 Analizar Sentimiento", variant="primary")
                    btn_limpiar = gr.Button("🗑️ Limpiar Pantalla")
                with gr.Column():
                    output_grafico = gr.Label(label="Resultado de la IA", num_top_classes=2)
                    with gr.Row():
                        btn_confirmar = gr.Button("✅ CONFIRMAR Y GUARDAR", variant="primary")
                        btn_corregir = gr.Button("❌ CORREGIR Y GUARDAR", variant="stop")
                    msg_status = gr.Markdown("")

        # PESTAÑA 2: CARGA MASIVA
        with gr.TabItem("📊 Carga Masiva (Empresarial)"):
            gr.Markdown("### 📂 Procesamiento de Lotes y Auto-Aprendizaje")
            with gr.Row():
                with gr.Column(scale=2):
                    file_input = gr.File(label="🚀 Arrastra tu Excel o CSV aquí", file_types=[".csv", ".xlsx"], height=250)
                    btn_batch = gr.Button("🚀 INICIAR PROCESAMIENTO MASIVO", variant="primary", size="lg")
                with gr.Column(scale=1):
                    gr.Markdown("#### 📈 Estado del Proceso:")
                    status_batch = gr.Markdown(value="### ⏳ Esperando archivo...", visible=True)
                    file_output = gr.File(label="📥 Descargar Resultados")

    # Lógica de botones
    btn_analizar.click(fn=predecir_solo_muestra, inputs=input_text, outputs=[output_grafico, msg_status])
    btn_limpiar.click(fn=limpiar_pantalla, inputs=None, outputs=[input_text, output_grafico, msg_status])

    # Nuevos botones de guardado controlado
    btn_confirmar.click(fn=confirmar_y_guardar, inputs=[input_text, output_grafico], outputs=msg_status)
    btn_corregir.click(fn=corregir_y_guardar, inputs=[input_text, output_grafico], outputs=msg_status)


    btn_batch.click(fn=procesar_archivo_masivo_automatizado, inputs=file_input, outputs=[file_output, status_batch])

    demo.launch(share=True, show_error=True)

Mounted at /content/drive
🤖 Activando ROBOT DE INTELIGENCIA REFORZADO...
✅ Robot reforzado R5K_v2 conectado y operativo


/tmp/ipython-input-302556680.py:211: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c59c258661d5d88c5a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime
from IPython.display import display

def monitorear_progreso_y_auditoria():
    """Genera un reporte visual y técnico del aprendizaje del modelo."""

    # Verificamos si existe el archivo de feedback (Aprendizaje Activo)
    if os.path.exists(DATASET_FEEDBACK):
        df_audit = pd.read_csv(DATASET_FEEDBACK)

        # --- 1. RESUMEN EJECUTIVO ---
        print(f"📊 PANEL DE CONTROL DE INTELIGENCIA (AUDITORÍA ACTIVA)")
        print(f"----------------------------------------------------")
        total = len(df_audit)
        conteos = df_audit['y'].value_counts().to_dict()
        pos = conteos.get(1, 0)
        neg = conteos.get(0, 0)

        print(f"🔹 Total de registros validados: {total}")
        print(f"✅ Sentimientos Positivos (1): {pos}")
        print(f"❌ Sentimientos Negativos (0): {neg}")

        # --- 2. VISUALIZACIÓN DE BALANCE ---
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

        # Gráfico de Barras: Volumen real por clase
        sns.barplot(x=['Negativo (0)', 'Positivo (1)'], y=[neg, pos],
                    palette=['#FF6B6B', '#4ECDC4'], ax=ax1)
        ax1.set_title("Volumen de Datos Validados (Historial)")
        for i, v in enumerate([neg, pos]):
            ax1.text(i, v + 0.1, str(v), ha='center', fontweight='bold')

        # Gráfico Circular: Proporción del dataset de entrenamiento
        if total > 0:
            ax2.pie([pos, neg], labels=['Positivos', 'Negativos'], autopct='%1.1f%%',
                    startangle=140, colors=['#4ECDC4', '#FF6B6B'], explode=(0.05, 0), shadow=True)
            ax2.set_title("Distribución del Conocimiento")

        plt.tight_layout()
        plt.show()

        # --- 3. AUDITORÍA DE PALABRAS CLAVE (INTERPRETABILIDAD) ---
        # Analizamos qué términos están moviendo la aguja del modelo tras las correcciones
        print("\n🔍 TOP 10 PALABRAS CLAVE (INFLUENCIA EN TIEMPO REAL):")
        try:
            # Extraemos coeficientes del modelo entrenado
            feature_names = vectorizer.get_feature_names_out()
            coefficients = model.coef_[0]
            features_df = pd.DataFrame({'Palabra': feature_names, 'Peso': coefficients})

            top_pos = features_df.sort_values(by='Peso', ascending=False).head(10)
            top_neg = features_df.sort_values(by='Peso', ascending=True).head(10)

            print("\n🟢 MÁS POSITIVAS (Empujan al 1):      🔴 MÁS NEGATIVAS (Empujan al 0):")
            for i in range(len(top_pos)):
                p_pos = f"{top_pos.iloc[i]['Palabra']}: {top_pos.iloc[i]['Peso']:.3f}"
                p_neg = f"{top_neg.iloc[i]['Palabra']}: {top_neg.iloc[i]['Peso']:.3f}"
                print(f"{p_pos.ljust(35)} {p_neg}")
        except NameError:
            print("⚠️ Modelo no cargado. Entrena el modelo para ver pesos de palabras.")

        # --- 4. TABLA DE AUDITORÍA CRUDA ---
        # Útil para detectar si se filtró algún duplicado erróneo
        print(f"\n📝 ÚLTIMOS 5 REGISTROS VALIDADOS EN EL SISTEMA:")
        display(df_audit.tail(5))

        # Alerta de calidad: detectar duplicados con etiquetas distintas
        duplicados = df_audit[df_audit.duplicated(subset=['clean_text'], keep=False)]
        if not duplicados.empty:
            frases_conflicto = duplicados.groupby('clean_text')['y'].nunique()
            conflictos_reales = frases_conflicto[frases_conflicto > 1]
            if not conflictos_reales.empty:
                print(f"\n⚠️ ALERTA DE CALIDAD: Se detectaron {len(conflictos_reales)} frases con etiquetas contradictorias.")
                print("Se recomienda limpiar el CSV para evitar confusión en la IA.")

    else:
        print("⚠️ No se encontró el archivo de auditoría. Realiza algunas validaciones en la interfaz primero.")

# Ejecutar el monitoreo consolidado
monitorear_progreso_y_auditoria()


⚠️ No se encontró el archivo de auditoría. Realiza algunas validaciones en la interfaz primero.
